In [1]:
# Leer el archivo XLS

import pandas as pd
import glob
import os

carpeta = r'data\valores_pozos'
ruta_xls = r'data\valores_caudalimetro\registro_caudalimetros.xlsx'

df_xls = pd.read_excel(ruta_xls)

# Inspeccionar
print(df_xls.dtypes)
print()
df_xls.head(10)

pozo                        int64
date_inicio_mes    datetime64[us]
caudalimetro                int64
Caudal                      int64
dtype: object



,pozo,date_inicio_mes,caudalimetro,Caudal
0,1,2026-06-01,1,20
1,2,2026-06-01,1,20
2,3,2026-06-01,1,20


In [2]:
archivos = glob.glob(os.path.join(carpeta, 'pozo_*.csv'))
print(f"Archivos encontrados: {len(archivos)}")

todos_los_ciclos = []

for archivo in archivos:
    nombre      = os.path.basename(archivo)
    numero_pozo = nombre.split('_')[1]

    df = pd.read_csv(archivo, sep=',')
    df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%y %H:%M:%S')
    df.rename(columns={df.columns[1]: 'estado'}, inplace=True)

    ciclos_este_archivo = 0
    for i in range(len(df) - 1):
# AHORA — solo ciclos mayores a 15 minutos:
        duracion_min = (df['Time'].iloc[i+1] - df['Time'].iloc[i]).total_seconds() / 60

        if df['estado'].iloc[i] == 1 and df['estado'].iloc[i+1] == 0 and duracion_min > 15:
            inicio = df['Time'].iloc[i]
            fin    = df['Time'].iloc[i+1]
            todos_los_ciclos.append({
                'pozo':  int(numero_pozo),
                'Fecha': f"{inicio.day}/{inicio.month:02d}/{inicio.year}",
                'ON':    inicio,    # datetime completo — NO strftime
                'OFF':   fin        # datetime completo — NO strftime
            })
            ciclos_este_archivo += 1

    print(f"✓ {nombre:30s} → {ciclos_este_archivo} ciclos")

print(f"\nTOTAL: {len(todos_los_ciclos)} ciclos")

Archivos encontrados: 3
✓ pozo_1_2026_06.csv             → 14 ciclos
✓ pozo_2_2026_06.csv             → 12 ciclos
✓ pozo_3_2026_06.csv             → 13 ciclos

TOTAL: 39 ciclos


In [3]:
df_final = pd.DataFrame(todos_los_ciclos)

# Verificar que ON y OFF son datetime con fecha real
print(df_final.dtypes)
print()
print(df_final[['pozo','Fecha','ON','OFF']].head(10))

pozo              int64
Fecha               str
ON       datetime64[us]
OFF      datetime64[us]
dtype: object

   pozo       Fecha                  ON                 OFF
0     1   1/06/2026 2026-06-01 06:01:47 2026-06-01 08:57:44
1     1   3/06/2026 2026-06-03 06:43:47 2026-06-03 13:33:01
2     1   5/06/2026 2026-06-05 11:02:01 2026-06-05 12:49:26
3     1   7/06/2026 2026-06-07 05:35:12 2026-06-07 11:17:51
4     1   9/06/2026 2026-06-09 11:14:28 2026-06-09 16:09:12
5     1  12/06/2026 2026-06-12 05:48:51 2026-06-12 08:01:12
6     1  14/06/2026 2026-06-14 09:09:13 2026-06-14 16:23:02
7     1  16/06/2026 2026-06-16 06:24:06 2026-06-16 09:51:47
8     1  18/06/2026 2026-06-18 09:51:02 2026-06-18 15:38:52
9     1  20/06/2026 2026-06-20 11:05:35 2026-06-20 14:08:24


In [4]:
ruta_xls = r'data\valores_caudalimetro\registro_caudalimetros.xlsx'

df_xls = pd.read_excel(ruta_xls)
df_xls['date_inicio_mes'] = pd.to_datetime(df_xls['date_inicio_mes'])
df_xls['anio'] = df_xls['date_inicio_mes'].dt.year
df_xls['mes']  = df_xls['date_inicio_mes'].dt.month
df_xls['pozo'] = df_xls['pozo'].astype(int)

print(df_xls.head(10))

   pozo date_inicio_mes  caudalimetro  Caudal  anio  mes
0     1      2026-06-01             1      20  2026    6
1     2      2026-06-01             1      20  2026    6
2     3      2026-06-01             1      20  2026    6


In [5]:
# Extraer año y mes del ON real
df_final['anio'] = df_final['ON'].dt.year
df_final['mes']  = df_final['ON'].dt.month

# Ordenar por pozo y por ON cronológico — crítico para la cadena
df_final = df_final.sort_values(by=['pozo', 'ON']).reset_index(drop=True)

# Agregar ID correlativo
df_final.insert(0, 'ID', range(1, len(df_final) + 1))

print(df_final[['ID','pozo','Fecha','ON','OFF','anio','mes']].head(10))

   ID  pozo       Fecha                  ON                 OFF  anio  mes
0   1     1   1/06/2026 2026-06-01 06:01:47 2026-06-01 08:57:44  2026    6
1   2     1   3/06/2026 2026-06-03 06:43:47 2026-06-03 13:33:01  2026    6
2   3     1   5/06/2026 2026-06-05 11:02:01 2026-06-05 12:49:26  2026    6
3   4     1   7/06/2026 2026-06-07 05:35:12 2026-06-07 11:17:51  2026    6
4   5     1   9/06/2026 2026-06-09 11:14:28 2026-06-09 16:09:12  2026    6
5   6     1  12/06/2026 2026-06-12 05:48:51 2026-06-12 08:01:12  2026    6
6   7     1  14/06/2026 2026-06-14 09:09:13 2026-06-14 16:23:02  2026    6
7   8     1  16/06/2026 2026-06-16 06:24:06 2026-06-16 09:51:47  2026    6
8   9     1  18/06/2026 2026-06-18 09:51:02 2026-06-18 15:38:52  2026    6
9  10     1  20/06/2026 2026-06-20 11:05:35 2026-06-20 14:08:24  2026    6


In [6]:
print(df_xls.columns)

Index(['pozo', 'date_inicio_mes', 'caudalimetro', 'Caudal', 'anio', 'mes'], dtype='str')


In [7]:
print(df_xls.columns.tolist())

['pozo', 'date_inicio_mes', 'caudalimetro', 'Caudal', 'anio', 'mes']


In [8]:
df_final['Lectura inicial'] = 0.0
df_final['Lectura final']   = 0.0
df_final['Caudal l/s']      = 0.0
df_final['m3']              = 0.0  # columna nueva
df_final['Hora ON']         = ''   # columna nueva — solo hora
df_final['Hora OFF']        = ''   # columna nueva — solo hora

ultima_lectura = {}

for idx in range(len(df_final)):

    pozo = df_final.loc[idx, 'pozo']
    anio = df_final.loc[idx, 'anio']
    mes  = df_final.loc[idx, 'mes']
    on   = df_final.loc[idx, 'ON']
    off  = df_final.loc[idx, 'OFF']

    # Buscar caudal en XLS
    fila_xls = df_xls[
        (df_xls['pozo'] == pozo) &
        (df_xls['anio'] == anio) &
        (df_xls['mes']  == mes)
    ]

    if len(fila_xls) == 0:
        print(f"⚠ Sin registro XLS: pozo {pozo}, {mes}/{anio}")
        continue

    caudal           = fila_xls['Caudal'].values[0]
    caudalimetro_mes = fila_xls['caudalimetro'].values[0]

    # Lectura inicial
    if pozo not in ultima_lectura:
        lectura_inicial = caudalimetro_mes
    else:
        lectura_inicial = ultima_lectura[pozo]

    # Calcular
          # Se usa round( operacion_matematica, cantidad_de_decimales )
    horas = round((off - on).total_seconds() / 3600, 2)
    m3 = horas * caudal * 3.6
    lectura_final = lectura_inicial + m3

    # Guardar
    df_final.loc[idx, 'Lectura inicial'] = round(lectura_inicial, 2)
    df_final.loc[idx, 'Lectura final']   = round(lectura_final, 2)
    df_final.loc[idx, 'Caudal l/s']      = caudal
    df_final.loc[idx, 'm3']              = round(m3, 2)
    df_final.loc[idx, 'Hora ON']         = on.strftime('%H:%M')   # solo hora
    df_final.loc[idx, 'Hora OFF']        = off.strftime('%H:%M')  # solo hora

    ultima_lectura[pozo] = lectura_final

print("✓ Cálculo completado")
df_final[['ID','pozo','Fecha', 'Hora ON', 'Hora OFF', 'ON','OFF','Lectura inicial','Lectura final', 'm3', 'Caudal l/s']].head(15)

✓ Cálculo completado


,ID,pozo,Fecha,Hora ON,Hora OFF,ON,OFF,Lectura inicial,Lectura final,m3,Caudal l/s
0,1,1,1/06/2026,06:01,08:57,2026-06-01 06:01:47,2026-06-01 08:57:44,1.00,211.96,210.96,20.0
1,2,1,3/06/2026,06:43,13:33,2026-06-03 06:43:47,2026-06-03 13:33:01,211.96,703.00,491.04,20.0
2,3,1,5/06/2026,11:02,12:49,2026-06-05 11:02:01,2026-06-05 12:49:26,703.00,831.88,128.88,20.0
3,4,1,7/06/2026,05:35,11:17,2026-06-07 05:35:12,2026-06-07 11:17:51,831.88,1243.00,411.12,20.0
4,5,1,9/06/2026,11:14,16:09,2026-06-09 11:14:28,2026-06-09 16:09:12,1243.00,1596.52,353.52,20.0
5,6,1,12/06/2026,05:48,08:01,2026-06-12 05:48:51,2026-06-12 08:01:12,1596.52,1755.64,159.12,20.0
6,7,1,14/06/2026,09:09,16:23,2026-06-14 09:09:13,2026-06-14 16:23:02,1755.64,2276.20,520.56,20.0
7,8,1,16/06/2026,06:24,09:51,2026-06-16 06:24:06,2026-06-16 09:51:47,2276.20,2525.32,249.12,20.0
8,9,1,18/06/2026,09:51,15:38,2026-06-18 09:51:02,2026-06-18 15:38:52,2525.32,2942.92,417.60,20.0
9,10,1,20/06/2026,11:05,14:08,2026-06-20 11:05:35,2026-06-20 14:08:24,2942.92,3162.52,219.60,20.0


In [9]:
# Eliminar columnas auxiliares
df_final = df_final.drop(columns=['anio', 'mes'])

# Orden final de columnas en el CSV
columnas_finales = [
    'ID', 'pozo', 'Fecha',
    'ON', 'OFF',            # datetime completo — para cálculos futuros
    'Hora ON', 'Hora OFF',  # solo hora — para leer fácil
    'Lectura inicial', 'Lectura final',
    'm3',
    'Caudal l/s'
]

df_final = df_final[columnas_finales]


ruta_salida = r'data\registro_pozos\registro_pozos_final.csv'
df_final.to_csv(ruta_salida, index=False)

print(f"✓ Guardado en: {ruta_salida}")
print(f"Total filas: {len(df_final)}")
df_final.head(10)

✓ Guardado en: data\registro_pozos\registro_pozos_final.csv
Total filas: 39


,ID,pozo,Fecha,ON,OFF,Hora ON,Hora OFF,Lectura inicial,Lectura final,m3,Caudal l/s
0,1,1,1/06/2026,2026-06-01 06:01:47,2026-06-01 08:57:44,06:01,08:57,1.00,211.96,210.96,20.0
1,2,1,3/06/2026,2026-06-03 06:43:47,2026-06-03 13:33:01,06:43,13:33,211.96,703.00,491.04,20.0
2,3,1,5/06/2026,2026-06-05 11:02:01,2026-06-05 12:49:26,11:02,12:49,703.00,831.88,128.88,20.0
3,4,1,7/06/2026,2026-06-07 05:35:12,2026-06-07 11:17:51,05:35,11:17,831.88,1243.00,411.12,20.0
4,5,1,9/06/2026,2026-06-09 11:14:28,2026-06-09 16:09:12,11:14,16:09,1243.00,1596.52,353.52,20.0
5,6,1,12/06/2026,2026-06-12 05:48:51,2026-06-12 08:01:12,05:48,08:01,1596.52,1755.64,159.12,20.0
6,7,1,14/06/2026,2026-06-14 09:09:13,2026-06-14 16:23:02,09:09,16:23,1755.64,2276.20,520.56,20.0
7,8,1,16/06/2026,2026-06-16 06:24:06,2026-06-16 09:51:47,06:24,09:51,2276.20,2525.32,249.12,20.0
8,9,1,18/06/2026,2026-06-18 09:51:02,2026-06-18 15:38:52,09:51,15:38,2525.32,2942.92,417.60,20.0
9,10,1,20/06/2026,2026-06-20 11:05:35,2026-06-20 14:08:24,11:05,14:08,2942.92,3162.52,219.60,20.0
